# Cross‑Dataset Benchmark: U‑MobileViT‑Net

**Mục tiêu**: Huấn luyện biến thể **Base** (0.6 M tham số) trên **6 dataset** đa dạng, đánh giá khả năng tổng quát hóa của kiến trúc.

**Datasets** (đa miền — nông nghiệp, đường phố, y tế):

| STT | Dataset       | Miền           | Loại          | Số lớp | Kích thước ảnh |
|-----|---------------|----------------|---------------|--------|----------------|
| 1   | COCO Tea Leaf | Nông nghiệp    | Multi‑class   | 8      | 320 × 320      |
| 2   | CamVid        | Đường phố      | Multi‑class   | 32     | 360 × 480      |
| 3   | Cityscapes    | Đường phố      | Multi‑class   | 19     | 512 × 1024     |
| 4   | PASCAL VOC    | Đa dụng        | Multi‑class   | 21     | 384 × 384      |
| 5   | Kvasir‑SEG    | Y tế (nội soi) | Binary        | 1      | 256 × 256      |
| 6   | ISIC 2018     | Y tế (da liễu) | Binary        | 1      | 256 × 256      |

**Metrics**:
- Multi‑class → **mIoU** (Mean Intersection over Union)
- Binary → **Dice Coefficient** (F1‑score)

---

## 1. Thiết lập Môi trường & Thư viện

In [ ]:
# ==============================================================================
# SECTION 1 — Environment Setup & Imports
# ==============================================================================

"""Thiết lập PYTHONPATH, CUDA config và import toàn bộ thư viện."""
import sys, os
from pathlib import Path

# Xác định project root (tìm .git / CLAUDE.md)
current = Path.cwd()
project_root = current
for parent in [current] + list(current.parents):
    if (parent / ".git").exists() or (parent / "CLAUDE.md").exists():
        project_root = parent
        break

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.environ.setdefault("PYTHONPATH", str(project_root))
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:128,garbage_collection_threshold:0.6")
os.chdir(str(project_root))
print(f"📁 Project root  : {project_root}")

import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
import seaborn as sns

from models.u_mobilevit_net.u_models import (
    umobilevit, UMobileViT, umobilevit_nano, umobilevit_base,
    umobilevit_pro, umobilevit_promax,
)
from models.u_mobilevit_net.configs import get_variant, UMOBILEVIT_VARIANTS
from tools.data import (
    create_dataloaders, DatasetInfo, label_to_color, denormalize,
)
from tools.training import (
    SegmentationTrainer, TrainingConfig,
    compute_class_weights, compute_pos_weight,
)
from tools.visualization import (
    configure_paper_style, plot_training_curves,
    show_dataset_samples, show_predictions,
    plot_per_class_iou, plot_confusion_matrix,
    plot_error_distribution, plot_params_vs_accuracy,
    plot_convergence_comparison, generate_results_table,
    load_training_history, load_all_results,
    TOL_PALETTE, VARIANT_COLORS,
)
from tools.evaluation import (
    compute_flops, compute_parameters, format_flops,
    compute_flops_by_component,
)

# Áp dụng style nhất quán cho publication
configure_paper_style()
# Mở rộng palette cho 6 dataset
sns.set_palette("husl", 6)

# Device detection
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Device        : {device}")
if device.type == "cuda":
    print(f"   GPU           : {torch.cuda.get_device_name(0)}")
    # print(f"   VRAM          : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
    print(f"   CUDA Version  : {torch.version.cuda}")


device, PROJECT_ROOT = device, project_root

📁 Project root  : /run/media/sanng/New Volume/Seminar/U-MOBILEVIT-NET
🖥️  Device        : cuda
   GPU           : NVIDIA GeForce RTX 4050 Laptop GPU


AttributeError: 'torch._C._CudaDeviceProperties' object has no attribute 'total_mem'

## 2. Cấu hình Benchmark

In [ ]:
# ==============================================================================
# SECTION 2 — Benchmark Configuration
# ==============================================================================

# Danh sách 6 dataset cần benchmark
BENCHMARK_DATASETS = ["coco_leaf", "camvid", "cityscapes", "voc", "kvasir", "isic"]

# Cấu hình huấn luyện riêng cho từng dataset
# Mỗi dataset có đặc thù riêng → hyperparameters được tối ưu riêng
DATASET_CONFIGS = {
    # ── Nông nghiệp ──
    "coco_leaf":  {
        "epochs": 300, "patience": 20,
        "image_size": (320, 320),
        "scheduler": "poly",  "class_weights": True,
        "label_smoothing": 0.1, "aug_intensity": "medium",
        "domain": "Agriculture",
    },
    # ── Đường phố ──
    "camvid":     {
        "epochs": 500, "patience": 20,
        "image_size": (360, 480),
        "scheduler": "poly",  "class_weights": True,
        "label_smoothing": 0.1, "aug_intensity": "strong",
        "domain": "Street Scene",
    },
    "cityscapes": {
        "epochs": 200, "patience": 20,
        "image_size": (512, 1024),
        "scheduler": "poly",  "class_weights": True,
        "label_smoothing": 0.1, "aug_intensity": "medium",
        "domain": "Street Scene",
    },
    # ── Đa dụng ──
    "voc":        {
        "epochs": 200, "patience": 20,
        "image_size": (384, 384),
        "scheduler": "poly",  "class_weights": True,
        "label_smoothing": 0.1, "aug_intensity": "strong",
        "domain": "General Object",
    },
    # ── Y tế ──
    "kvasir":     {
        "epochs": 150, "patience": 15,
        "image_size": (256, 256),
        "scheduler": "cosine", "class_weights": False,
        "label_smoothing": 0.0, "aug_intensity": "medium",
        "domain": "Medical (Endoscopy)",
    },
    "isic":       {
        "epochs": 150, "patience": 15,
        "image_size": (256, 256),
        "scheduler": "cosine", "class_weights": False,
        "label_smoothing": 0.0, "aug_intensity": "strong",
        "domain": "Medical (Dermatology)",
    },
}

# Tên hiển thị cho biểu đồ
DISPLAY_NAMES = {
    "coco_leaf": "COCO Tea Leaf",
    "camvid": "CamVid",
    "cityscapes": "Cityscapes",
    "voc": "PASCAL VOC\n2012",
    "kvasir": "Kvasir‑SEG",
    "isic": "ISIC 2018",
}

VARIANT = "base"

print("╔══════════════════════════════════════════════════════════════════╗")
print(f"║   BENCHMARK CONFIGURATION                                       ║")
print(f"║   Variant  : {VARIANT.upper():<50s}║")
print(f"║   Datasets : {len(BENCHMARK_DATASETS):<50d}║")
print("╚══════════════════════════════════════════════════════════════════╝")
print()
for ds in BENCHMARK_DATASETS:
    cfg = DATASET_CONFIGS[ds]
    print(f"  {DISPLAY_NAMES[ds]:<25s}  "
          f"epochs={cfg['epochs']:>4d}  "
          f"size={str(cfg['image_size']):<12s}  "
          f"sched={cfg['scheduler']:<6s}  "
          f"domain={cfg['domain']}")


## 3. Khởi tạo Mô hình & Phân tích Kiến trúc

In [ ]:
# ==============================================================================
# SECTION 3 — Model Initialization & FLOPs/Params Analysis
# ==============================================================================

# Tạo một instance mẫu để phân tích (dùng số lớp của dataset đầu tiên làm mẫu)
sample_model = umobilevit_base(out_channels=8, head="single")
sample_model = sample_model.to(device)

params_total = compute_parameters(sample_model)
flops_total, flops_breakdown = compute_flops(sample_model, input_size=(320, 320))
flops_by_component = compute_flops_by_component(sample_model, (320, 320))

print("\n╔══════════════════════════════════════════════════════════════════╗")
print(f"║   MODEL: U-MobileViT-Net [{VARIANT.upper():^5s}]                                  ║")
print("╠══════════════════════════════════════════════════════════════════╣")
print(f"║   Total Parameters : {params_total/1e6:>8.2f} M                                ║")
print(f"║   Total FLOPs      : {format_flops(flops_total):>10s}                               ║")
print("╠══════════════════════════════════════════════════════════════════╣")
print("║   FLOPs Breakdown (320×320):                                    ║")
for comp, f in sorted(flops_by_component.items(), key=lambda x: -x[1]):
    pct = 100 * f / flops_total
    bar = "█" * int(pct / 2)
    print(f"║     {comp:<14s}  {format_flops(f):>8s}  ({pct:5.1f}%)  {bar}║")
print("╚══════════════════════════════════════════════════════════════════╝")

# Clean up sample
import numpy as np
del sample_model
if device.type == "cuda":
    torch.cuda.empty_cache()

## 4. Vòng lặp Huấn luyện — Từng Dataset

Mỗi dataset được huấn luyện **độc lập** với cấu hình tối ưu riêng.  
Kết quả được lưu vào `checkpoints/<dataset>/training_history.json`.

In [ ]:
# ==============================================================================
# SECTION 4 — Training Loop (All Datasets)
# ==============================================================================

def train_on_dataset(ds_name: str, variant: str, device: torch.device):
    """Huấn luyện U‑MobileViT‑Net trên MỘT dataset và trả về kết quả."""
    import os
    from tools.data import create_dataloaders
    from tools.training import SegmentationTrainer, TrainingConfig
    from tools.training import compute_class_weights, compute_pos_weight

    cfg = DATASET_CONFIGS[ds_name]
    display_name = DISPLAY_NAMES[ds_name]

    print(f"\n{'═' * 70}")
    print(f"  📦 DATASET: {display_name}  |  Domain: {cfg['domain']}")
    print(f"  🎯 Classes: TBD  |  Scheduler: {cfg['scheduler']}  |  LR: 1e-3")
    print(f"{'═' * 70}")

    # ── Tạo DataLoaders ──
    print(f"  [1/5] Loading data (aug={cfg['aug_intensity']})...")
    train_loader, val_loader, info = create_dataloaders(
        ds_name,
        image_size=cfg["image_size"],
        batch_size=16,
        num_workers=8,
        aug_intensity=cfg["aug_intensity"],
    )
    print(f"        Train: {len(train_loader.dataset):,d} samples  |  "
          f"Val: {len(val_loader.dataset):,d} samples")
    print(f"        Classes: {info.num_classes}  |  Type: {info.type}")

    # ── Khởi tạo model ──
    print(f"  [2/5] Building model...")
    factory = {"base": umobilevit_base}
    model = factory[variant](out_channels=info.num_classes, head="single")
    model = model.to(device)
    params_m = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
    print(f"        Parameters: {params_m:.2f} M")

    # ── Class weights (nếu cần) ──
    print(f"  [3/5] Computing class weights...")
    class_weights = None
    pos_weight = None
    if cfg["class_weights"] and info.type == "multi-class":
        class_weights = compute_class_weights(
            train_loader, info.num_classes,
            ignore_index=info.ignore_index, device=device,
        )
        print(f"        Class weights: min={class_weights.min().item():.3f}, "
              f"max={class_weights.max().item():.3f}")
    elif info.type == "binary":
        pos_weight = compute_pos_weight(train_loader, device=device)
        print(f"        Pos weight: {pos_weight.item():.3f}" if pos_weight is not None
              else "        No class weighting")

    # ── Trainer config ──
    save_dir = f"./checkpoints/{ds_name}"
    os.makedirs(save_dir, exist_ok=True)

    trainer = SegmentationTrainer(
        model=model,
        device=device,
        dataset_info=info,
        save_dir=save_dir,
        config=TrainingConfig(
            lr=1e-3,
            weight_decay=1e-4,
            warmup_epochs=5,
            scheduler_type=cfg["scheduler"],
            label_smoothing=cfg["label_smoothing"],
        ),
        class_weights=class_weights,
        pos_weight=pos_weight,
    )

    # ── Train ──
    print(f"  [4/5] Training ({cfg['epochs']} epochs, patience={cfg['patience']})...")
    history = trainer.train(
        train_loader, val_loader,
        epochs=cfg["epochs"],
        patience=cfg["patience"],
        min_epochs=30,
    )

    # ── Kết quả ──
    metric_name = "Dice" if info.type == "binary" else "mIoU"
    best_metric = max(history["val_metric"])
    best_epoch = history["val_metric"].index(best_metric) + 1
    print(f"  [5/5] ✅ Best {metric_name}: {best_metric:.4f}  (epoch {best_epoch})")

    result = {
        "history": history,
        "info": info,
        "trainer": trainer,
        "best_metric": best_metric,
        "best_epoch": best_epoch,
        "metric_name": metric_name,
        "params_m": params_m,
        "flops_m": sum(
            compute_flops(model, cfg["image_size"])[0]
        ) / 1e6 if hasattr(cfg, '__iter__') else 0,
    }

    # Cleanup GPU memory
    del model, trainer
    if device.type == "cuda":
        torch.cuda.empty_cache()

    return result


# ── Chạy benchmark tuần tự ──
all_results = {}
for idx, ds_name in enumerate(BENCHMARK_DATASETS, 1):
    print(f"\n{'▸' * 35}")
    print(f"   BENCHMARK [{idx}/{len(BENCHMARK_DATASETS)}]")
    print(f"{'▸' * 35}")

    result = train_on_dataset(ds_name, VARIANT, device)
    all_results[ds_name] = result

print(f"\n{'█' * 70}")
print(f"  ✅ ALL {len(BENCHMARK_DATASETS)} DATASETS COMPLETED")
print(f"{'█' * 70}")


## 5. Tổng quan Kết quả — Bảng So sánh Chi tiết

In [ ]:
# ==============================================================================
# SECTION 5 — Detailed Comparison Table
# ==============================================================================

import pandas as pd
from IPython.display import display, HTML

def build_comparison_dataframe(all_results):
    """Xây dựng DataFrame so sánh chi tiết với styling."""
    rows = []
    for ds_name in BENCHMARK_DATASETS:
        if ds_name not in all_results:
            continue
        res = all_results[ds_name]
        cfg = DATASET_CONFIGS[ds_name]
        rows.append({
            "Dataset": DISPLAY_NAMES[ds_name],
            "Domain": cfg["domain"],
            "Classes": res["info"].num_classes,
            "Type": res["info"].type,
            "Metric": res["metric_name"],
            "Best Score": res["best_metric"],
            "Best Epoch": res["best_epoch"],
            "Total Epochs": min(
                len(res["history"]["val_metric"]), cfg["epochs"]
            ),
            "Image Size": f"{cfg['image_size'][0]}×{cfg['image_size'][1]}",
            "Augmentation": cfg["aug_intensity"].capitalize(),
        })

    df = pd.DataFrame(rows)
    df = df.set_index("Dataset")
    # Sắp xếp theo domain rồi score
    df = df.sort_values(["Domain", "Best Score"], ascending=[True, False])
    return df

# ── Hiển thị bảng ──
print("\n╔══════════════════════════════════════════════════════════════════════════════╗")
print(f"║   CROSS‑DATASET BENCHMARK  |  U‑MobileViT‑Net [{VARIANT.upper()}]                         ║")
print("╚══════════════════════════════════════════════════════════════════════════════╝")
print()

# Bảng thô (pure text)
print(f"  {'Dataset':<25s}  {'Domain':<22s}  {'Cls':>4s}  {'Metric':>6s}  {'Score':>8s}  {'Ep':>5s}")
print(f"  {'─' * 80}")
for ds_name in BENCHMARK_DATASETS:
    if ds_name not in all_results:
        continue
    res = all_results[ds_name]
    cfg = DATASET_CONFIGS[ds_name]
    print(f"  {DISPLAY_NAMES[ds_name]:<25s}  "
          f"{cfg['domain']:<22s}  "
          f"{res['info'].num_classes:>4d}  "
          f"{res['metric_name']:>6s}  "
          f"{res['best_metric']:>8.4f}  "
          f"{res['best_epoch']:>5d}")
print(f"{'═' * 80}")

# DataFrame với styling màu
df_comparison = build_comparison_dataframe(all_results)

# Highlight cột score theo giá trị
def color_score(val):
    """Tô màu gradient cho cột score."""
    if pd.isna(val):
        return ""
    # Green intensity proportional to value
    intensity = int(val * 255)
    return f"background-color: rgba(39, 174, 96, {val:.2f}); color: #1a1a2e"

styled = df_comparison.style \
    .format({"Best Score": "{:.4f}"}) \
    .applymap(color_score, subset=["Best Score"]) \
    .set_caption(f"U‑MobileViT‑Net [{VARIANT.upper()}] — Cross‑Dataset Benchmark Results")

display(styled)


## 6. Trực quan hóa ① — Biểu đồ Cột So sánh Điểm số

In [ ]:
# ==============================================================================
# SECTION 6 — Bar Chart: Best Metric per Dataset
# ==============================================================================

fig, ax = plt.subplots(figsize=(14, 6))

# Chuẩn bị dữ liệu
ds_labels = []
scores = []
colors = []
domains = []

# Color map theo domain
DOMAIN_COLORS = {
    "Agriculture": TOL_PALETTE[0],         # blue
    "Street Scene": TOL_PALETTE[3],         # red
    "General Object": TOL_PALETTE[1],       # green
    "Medical (Endoscopy)": TOL_PALETTE[5],  # cyan
    "Medical (Dermatology)": TOL_PALETTE[4],# purple
}

for ds_name in BENCHMARK_DATASETS:
    if ds_name not in all_results:
        continue
    res = all_results[ds_name]
    cfg = DATASET_CONFIGS[ds_name]
    ds_labels.append(DISPLAY_NAMES[ds_name].replace("\n", " "))
    scores.append(res["best_metric"])
    colors.append(DOMAIN_COLORS.get(cfg["domain"], TOL_PALETTE[6]))
    domains.append(cfg["domain"])

# Vẽ bar chart
bars = ax.bar(range(len(ds_labels)), scores, color=colors,
              edgecolor="white", linewidth=1.2, width=0.6)

# Thêm giá trị trên mỗi cột
for i, (bar, score, ds_name) in enumerate(zip(bars, scores, ds_labels)):
    res = all_results[BENCHMARK_DATASETS[i]]
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.012,
        f"{score:.4f}\n({res['metric_name']})",
        ha="center", va="bottom", fontsize=9, fontweight="bold",
        color="#1a1a2e",
    )

# Trang trí trục
ax.set_xticks(range(len(ds_labels)))
ax.set_xticklabels(ds_labels, fontsize=10, rotation=25, ha="right")
ax.set_ylabel("Best Validation Score", fontsize=12)
ax.set_ylim(0, max(scores) * 1.18)
ax.set_title(
    f"U‑MobileViT‑Net [{VARIANT.upper()}] — Cross‑Dataset Performance\n"
    f"Metric: mIoU (multi‑class) / Dice (binary)",
    fontsize=13, fontweight="bold", pad=15,
)
ax.grid(True, axis="y", ls="--", alpha=0.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Legend cho domain
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=c, edgecolor="white", label=d)
    for d, c in DOMAIN_COLORS.items()
    if d in domains
]
ax.legend(handles=legend_elements, loc="lower right",
          framealpha=0.9, fontsize=8, title="Domain")

plt.tight_layout()
os.makedirs("paper_figures", exist_ok=True)
fig.savefig("paper_figures/02_benchmark_barchart.pdf")
plt.show()


## 7. Trực quan hóa ② — Đường cong Hội tụ (Convergence)

In [ ]:
# ==============================================================================
# SECTION 7 — Convergence Curves (2×3 Grid)
# ==============================================================================

def plot_convergence_grid(all_results, variant="base"):
    """Vẽ lưới 2×3 đường cong hội tụ val_metric cho 6 dataset."""
    n = len(all_results)
    cols = 3
    rows = 2

    fig, axes = plt.subplots(rows, cols, figsize=(18, 10))
    axes = axes.flatten()

    base_color = VARIANT_COLORS.get(variant, TOL_PALETTE[1])

    for idx, ds_name in enumerate(BENCHMARK_DATASETS):
        if ds_name not in all_results:
            axes[idx].set_visible(False)
            continue

        ax = axes[idx]
        res = all_results[ds_name]
        history = res["history"]
        cfg = DATASET_CONFIGS[ds_name]

        epochs = range(1, len(history["val_metric"]) + 1)
        metric_name = res["metric_name"]
        best_val = res["best_metric"]
        best_ep = res["best_epoch"]

        # Val metric (đường chính)
        ax.plot(epochs, history["val_metric"],
                color=base_color, lw=2.5, label=f"Val {metric_name}", zorder=3)
        # Train loss (đường mờ, trục phải)
        ax2 = ax.twinx()
        ax2.plot(epochs, history["train_loss"],
                 color=TOL_PALETTE[3], lw=0.8, alpha=0.35, label="Train Loss")
        ax2.set_ylabel("Train Loss", fontsize=7, color=TOL_PALETTE[3])
        ax2.tick_params(axis="y", labelsize=6, colors=TOL_PALETTE[3])

        # Đánh dấu điểm tốt nhất
        ax.scatter([best_ep], [best_val],
                   color=TOL_PALETTE[2], s=60, zorder=5, edgecolors="white",
                   linewidth=0.8)
        ax.annotate(
            f"{best_val:.4f}",
            (best_ep, best_val),
            textcoords="offset points",
            xytext=(8, 8),
            fontsize=8,
            fontweight="bold",
            color=TOL_PALETTE[2],
            bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.8),
        )

        # Tiêu đề dataset
        ax.set_title(
            f"{DISPLAY_NAMES[ds_name].replace(chr(10), ' ')}  |  "
            f"{cfg['domain']}\nBest {metric_name} = {best_val:.4f}",
            fontsize=9, fontweight="bold",
        )
        ax.set_xlabel("Epoch", fontsize=8)
        ax.set_ylabel(metric_name, fontsize=8, color=base_color)
        ax.tick_params(axis="y", labelsize=7, colors=base_color)
        ax.tick_params(axis="x", labelsize=7)
        ax.grid(True, ls="--", alpha=0.4)
        ax.legend(loc="lower right", fontsize=6)

    # Ẩn subplot thừa
    for idx in range(n, rows * cols):
        axes[idx].set_visible(False)

    fig.suptitle(
        f"U‑MobileViT‑Net [{variant.upper()}] — Convergence on 6 Benchmark Datasets",
        fontsize=14, fontweight="bold", y=1.01,
    )
    plt.tight_layout()
    os.makedirs("paper_figures", exist_ok=True)
    fig.savefig("paper_figures/02_benchmark_convergence_grid.pdf")
    plt.show()

plot_convergence_grid(all_results, VARIANT)


## 8. Trực quan hóa ③ — Loss Curves (Từng Dataset)

In [ ]:
# ==============================================================================
# SECTION 8 — Detailed Loss Curves (per dataset)
# ==============================================================================

def plot_detailed_loss_curves(all_results):
    """Vẽ loss curves chi tiết cho từng dataset: CE/BCE loss, Dice loss, LR schedule."""
    n = len(all_results)
    cols = 3
    rows = 2

    fig, axes = plt.subplots(rows, cols, figsize=(18, 10))
    axes = axes.flatten()

    for idx, ds_name in enumerate(BENCHMARK_DATASETS):
        if ds_name not in all_results:
            axes[idx].set_visible(False)
            continue

        ax = axes[idx]
        res = all_results[ds_name]
        history = res["history"]

        epochs = range(1, len(history["val_metric"]) + 1)

        # Dice loss
        ax.plot(epochs, history["val_dice"],
                color=TOL_PALETTE[0], lw=1.8, label="Val Dice Loss")
        # CE loss
        ax.plot(epochs, history["val_ce"],
                color=TOL_PALETTE[3], lw=1.8, label="Val CE Loss")

        # LR on twin axis
        if history.get("lr"):
            ax_lr = ax.twinx()
            ax_lr.plot(epochs, history["lr"],
                       color=TOL_PALETTE[5], ls="--", lw=1, alpha=0.6, label="LR")
            ax_lr.set_ylabel("Learning Rate", fontsize=7, color=TOL_PALETTE[5])
            ax_lr.tick_params(axis="y", labelsize=6, colors=TOL_PALETTE[5])
            ax_lr.set_yscale("log")

        # Đánh dấu điểm best metric
        best_ep = res["best_epoch"]
        ax.axvline(best_ep, color=TOL_PALETTE[1], ls="--", lw=1, alpha=0.5,
                   label=f"Best Ep={best_ep}")

        ax.set_title(f"{DISPLAY_NAMES[ds_name].replace(chr(10), ' ')}",
                     fontsize=9, fontweight="bold")
        ax.set_xlabel("Epoch", fontsize=8)
        ax.set_ylabel("Loss", fontsize=8)
        ax.legend(loc="upper right", fontsize=6, ncol=2)
        ax.grid(True, ls="--", alpha=0.3)

    for idx in range(n, rows * cols):
        axes[idx].set_visible(False)

    fig.suptitle("Detailed Loss Components Across Datasets",
                 fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    os.makedirs("paper_figures", exist_ok=True)
    fig.savefig("paper_figures/02_benchmark_loss_curves.pdf")
    plt.show()

plot_detailed_loss_curves(all_results)


## 9. Trực quan hóa ④ — Heatmap So sánh Tương quan

In [ ]:
# ==============================================================================
# SECTION 9 — Correlation Heatmap (Dataset Similarity)
# ==============================================================================

def plot_dataset_heatmap(all_results):
    """Vẽ heatmap thể hiện mối tương quan giữa các dataset dựa trên đặc điểm."""
    # Xây dựng ma trận đặc trưng: [num_classes, image_size_area, epochs, score]
    features = {}
    for ds_name in BENCHMARK_DATASETS:
        if ds_name not in all_results:
            continue
        res = all_results[ds_name]
        cfg = DATASET_CONFIGS[ds_name]
        features[ds_name] = {
            "num_classes": res["info"].num_classes,
            "image_area": cfg["image_size"][0] * cfg["image_size"][1],
            "epochs": cfg["epochs"],
            "score": res["best_metric"],
        }

    ds_list = list(features.keys())
    n = len(ds_list)

    # Ma trận tương quan dựa trên "khoảng cách" giữa các dataset
    # (đơn giản: normalized Euclidean distance → similarity)
    import numpy as np

    feat_matrix = np.array([
        [
            features[ds]["num_classes"] / 32.0,
            features[ds]["image_area"] / (1024 * 512),
            features[ds]["epochs"] / 500.0,
            features[ds]["score"],
        ]
        for ds in ds_list
    ])

    # Cosine similarity
    from sklearn.metrics.pairwise import cosine_similarity
    sim_matrix = cosine_similarity(feat_matrix)

    labels = [DISPLAY_NAMES[ds].replace("\n", " ") for ds in ds_list]

    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(sim_matrix, cmap="RdYlGn", vmin=0.5, vmax=1.0, aspect="auto")

    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=9)
    ax.set_yticklabels(labels, fontsize=9)

    # Ghi giá trị vào từng ô
    for i in range(n):
        for j in range(n):
            ax.text(
                j, i, f"{sim_matrix[i, j]:.2f}",
                ha="center", va="center",
                fontsize=8,
                color="white" if sim_matrix[i, j] < 0.75 else "#1a1a2e",
            )

    ax.set_title(
        "Dataset Similarity Heatmap\n(based on class count, image size, training epochs, score)",
        fontsize=12, fontweight="bold", pad=12,
    )

    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Cosine Similarity", fontsize=9)

    plt.tight_layout()
    os.makedirs("paper_figures", exist_ok=True)
    fig.savefig("paper_figures/02_benchmark_heatmap.pdf")
    plt.show()

# Kiểm tra xem sklearn có sẵn không
try:
    plot_dataset_heatmap(all_results)
except ImportError:
    print("⚠️  sklearn not available — skipping heatmap.")


## 10. Trực quan hóa ⑤ — Radar Chart: So sánh Đa chiều

In [ ]:
# ==============================================================================
# SECTION 10 — Radar / Spider Chart
# ==============================================================================

def plot_radar_chart(all_results):
    """Vẽ radar chart thể hiện hiệu năng trên 6 dataset trong cùng một biểu đồ tròn."""
    import numpy as np

    ds_list = [ds for ds in BENCHMARK_DATASETS if ds in all_results]
    n = len(ds_list)
    if n < 3:
        print("⚠️  Need ≥3 datasets for radar chart.")
        return

    scores = [all_results[ds]["best_metric"] for ds in ds_list]
    labels = [DISPLAY_NAMES[ds].replace("\n", " ") for ds in ds_list]

    # Góc cho mỗi trục
    angles = np.linspace(0, 2 * np.pi, n, endpoint=False).tolist()
    scores += scores[:1]  # Đóng vòng
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

    ax.fill(angles, scores, color=TOL_PALETTE[1], alpha=0.25)
    ax.plot(angles, scores, color=TOL_PALETTE[1], lw=2.5, marker="o",
            markersize=8, markeredgecolor="white", markeredgewidth=1.2)

    # Ghi giá trị tại mỗi điểm
    for angle, score, label in zip(angles[:-1], scores[:-1], labels):
        ax.annotate(
            f"{score:.4f}",
            xy=(angle, score),
            xytext=(10, 10),
            textcoords="offset points",
            fontsize=8,
            fontweight="bold",
            color=TOL_PALETTE[0],
            bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.8),
        )

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylim(0, 1.0)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels(["0.2", "0.4", "0.6", "0.8", "1.0"], fontsize=7, color="gray")
    ax.set_title(
        f"U‑MobileViT‑Net [{VARIANT.upper()}] — Multi‑Dimensional Performance Radar",
        fontsize=13, fontweight="bold", pad=25,
    )
    ax.grid(True, ls="--", alpha=0.4)

    plt.tight_layout()
    os.makedirs("paper_figures", exist_ok=True)
    fig.savefig("paper_figures/02_benchmark_radar.pdf")
    plt.show()

plot_radar_chart(all_results)


## 11. Trực quan hóa ⑥ — Biểu đồ Miền: Epochs & Điểm số

In [ ]:
# ==============================================================================
# SECTION 11 — Scatter: Training Efficiency (Epochs vs Score)
# ==============================================================================

fig, ax = plt.subplots(figsize=(12, 7))

for idx, ds_name in enumerate(BENCHMARK_DATASETS):
    if ds_name not in all_results:
        continue
    res = all_results[ds_name]
    cfg = DATASET_CONFIGS[ds_name]
    color = DOMAIN_COLORS.get(cfg["domain"], TOL_PALETTE[6])

    # Kích thước điểm tượng trưng cho số class
    size = max(100, res["info"].num_classes * 25)

    ax.scatter(
        [res["best_epoch"]], [res["best_metric"]],
        s=size, c=color, edgecolors="white", linewidth=1.5,
        zorder=3, alpha=0.85,
    )
    ax.annotate(
        DISPLAY_NAMES[ds_name].replace("\n", " "),
        (res["best_epoch"], res["best_metric"]),
        textcoords="offset points",
        xytext=(10, 5),
        fontsize=9,
        fontweight="bold",
        color=color,
    )

ax.set_xlabel("Epoch đạt Best Metric", fontsize=12)
ax.set_ylabel("Best Score (mIoU / Dice)", fontsize=12)
ax.set_title(
    f"U‑MobileViT‑Net [{VARIANT.upper()}] — Training Efficiency\n"
    "(bubble size ∝ number of classes)",
    fontsize=13, fontweight="bold",
)
ax.grid(True, ls="--", alpha=0.4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Legend cho domain
legend_elements = [
    plt.scatter([], [], s=100, c=c, edgecolors="white", linewidth=1,
                label=d, alpha=0.85)
    for d, c in DOMAIN_COLORS.items()
    if d in domains
]
ax.legend(handles=legend_elements, loc="lower right",
          framealpha=0.9, fontsize=8, title="Domain")

plt.tight_layout()
os.makedirs("paper_figures", exist_ok=True)
fig.savefig("paper_figures/02_benchmark_efficiency.pdf")
plt.show()


## 12. Tóm tắt & Kết luận

In [ ]:
# ==============================================================================
# SECTION 12 — Summary & Conclusions
# ==============================================================================

print("\n" + "█" * 70)
print("  ✅ BENCHMARK HOÀN THÀNH — U‑MobileViT‑Net [{}]".format(VARIANT.upper()))
print("█" * 70)
print()
print(f"  {'Dataset':<25s}  {'Domain':<25s}  {'Metric':>6s}  {'Score':>8s}")
print(f"  {'─' * 70}")

all_scores = []
for ds_name in BENCHMARK_DATASETS:
    if ds_name not in all_results:
        continue
    res = all_results[ds_name]
    cfg = DATASET_CONFIGS[ds_name]
    all_scores.append(res["best_metric"])
    print(f"  {DISPLAY_NAMES[ds_name]:<25s}  "
          f"{cfg['domain']:<25s}  "
          f"{res['metric_name']:>6s}  "
          f"{res['best_metric']:>8.4f}")

print(f"  {'─' * 70}")
print(f"  {'Average Score':<25s}  {'':<25s}  {'':>6s}  {np.mean(all_scores):>8.4f}")
print(f"  {'Std Deviation':<25s}  {'':<25s}  {'':>6s}  {np.std(all_scores):>8.4f}")
print()

# ── Tổng hợp nhận xét ──
print("─" * 70)
print("  📊 KEY OBSERVATIONS:")
print()

# Tìm dataset tốt nhất / tệ nhất
best_ds = BENCHMARK_DATASETS[np.argmax(all_scores)]
worst_ds = BENCHMARK_DATASETS[np.argmin(all_scores)]

print(f"  1. Best performance  : {DISPLAY_NAMES[best_ds]}"
      f" ({all_results[best_ds]['best_metric']:.4f} {all_results[best_ds]['metric_name']})")
print(f"  2. Most challenging  : {DISPLAY_NAMES[worst_ds]}"
      f" ({all_results[worst_ds]['best_metric']:.4f} {all_results[worst_ds]['metric_name']})")
print(f"  3. Average score     : {np.mean(all_scores):.4f} ± {np.std(all_scores):.4f}")
print()

# Phân tích theo domain
from collections import defaultdict
domain_scores = defaultdict(list)
for ds_name in BENCHMARK_DATASETS:
    if ds_name not in all_results:
        continue
    cfg = DATASET_CONFIGS[ds_name]
    domain_scores[cfg["domain"]].append(all_results[ds_name]["best_metric"])

print("  Performance by Domain:")
for domain, scores_list in sorted(domain_scores.items()):
    print(f"    • {domain:<28s}: "
          f"mean={np.mean(scores_list):.4f}  "
          f"(n={len(scores_list)})")

print()
print(f"  💾 All checkpoints saved under: ./checkpoints/")
print(f"  📈 All figures saved under   : ./paper_figures/")
print("█" * 70)


---

## Phụ lục: Xuất Bảng LaTeX cho Paper

Cell bên dưới sinh mã LaTeX cho bảng kết quả benchmark để chèn trực tiếp vào bài báo.

In [ ]:
# ==============================================================================
# APPENDIX — LaTeX Table Export
# ==============================================================================

# Dựng dict {variant: {dataset: best_metric}} cho generate_results_table
variant_results = {VARIANT: {}}
for ds_name in BENCHMARK_DATASETS:
    if ds_name in all_results:
        variant_results[VARIANT][ds_name] = all_results[ds_name]["best_metric"]

# Metric map
metric_map = {}
for ds_name in BENCHMARK_DATASETS:
    if ds_name in all_results:
        metric_map[ds_name] = all_results[ds_name]["metric_name"]

latex_source = generate_results_table(
    variant_results,
    model_params={VARIANT: all_results[BENCHMARK_DATASETS[0]]["params_m"]},
    metric_map=metric_map,
    save_path="paper_figures/02_benchmark_table.tex",
)

print("LaTeX table exported to: paper_figures/02_benchmark_table.tex")
print()
print("─" * 50)
print(latex_source)
print("─" * 50)
